# Notebook 02 — Authoritative neutral contract and Telecom Pack

This notebook creates the **locked vocabulary** for Milestone 1:

- **SPEC-CORE v0.3**: information a detector may observe at runtime.
- **SPEC-EVAL v0.3**: fault and condition truth used only after scoring.
- **Telecom Pack v0.1**: telecom metric, hierarchy, quality and exposure meaning.
- A hash-pinned Python source directory used by Notebooks 03–05.

It does not translate data. Run it once before the other notebooks.

In [ ]:
from pathlib import Path
import json
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_DRIVE_ROOT = Path("/content/drive/MyDrive/anomaly_detection")
else:
    DEFAULT_DRIVE_ROOT = Path.cwd() / "anomaly_detection"

DRIVE_ROOT = Path(
    os.environ.get("ANOMALY_DETECTION_DRIVE_ROOT", str(DEFAULT_DRIVE_ROOT))
).expanduser()
CONTRACT_TAG = "v0.3"
CONTRACT_ROOT = DRIVE_ROOT / "contracts" / CONTRACT_TAG
PYTHON_SOURCE_ROOT = CONTRACT_ROOT / "python_src"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "milestone_1" / CONTRACT_TAG

print("Drive root:   ", DRIVE_ROOT)
print("Contract root:", CONTRACT_ROOT)
print("Output root:  ", OUTPUT_ROOT)

## 1. Publish the frozen implementation

The implementation is supplied as the separate, reviewable
`M1_v0_3_runtime_bundle.zip` file included with these notebooks. Put that file at:

`MyDrive/anomaly_detection/bootstrap/M1_v0_3_runtime_bundle.zip`

The short installer below verifies both the bundle and every extracted file.
Re-running is safe when contents are identical; changed content is never overwritten.

In [ ]:
import hashlib
import zipfile
from pathlib import PurePosixPath

RUNTIME_BUNDLE_NAME = "M1_v0_3_runtime_bundle.zip"
EXPECTED_BUNDLE_SHA256 = "6f81eb057e9f155a7f436d9520c30dd1107ecbcc53ffc4b87eff087c1b450b0f"

configured_bundle = os.environ.get("ANOMALY_DETECTION_RUNTIME_BUNDLE")
bundle_candidates = []
if configured_bundle:
    bundle_candidates.append(Path(configured_bundle).expanduser())
bundle_candidates.extend([
    DRIVE_ROOT / "bootstrap" / RUNTIME_BUNDLE_NAME,
    DRIVE_ROOT / RUNTIME_BUNDLE_NAME,
    Path("/content") / RUNTIME_BUNDLE_NAME,
])
RUNTIME_BUNDLE = next(
    (path for path in bundle_candidates if path.is_file()),
    None,
)
if RUNTIME_BUNDLE is None:
    raise FileNotFoundError(
        "Runtime bundle not found. Copy M1_v0_3_runtime_bundle.zip to "
        "MyDrive/anomaly_detection/bootstrap/ and rerun this cell."
    )

def write_immutable_text(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        if existing != text:
            raise FileExistsError(f"Immutable artifact differs: {path}")
        return "verified"
    path.write_text(text, encoding="utf-8")
    return "created"

actual_bundle_hash = hashlib.sha256(RUNTIME_BUNDLE.read_bytes()).hexdigest()
assert actual_bundle_hash == EXPECTED_BUNDLE_SHA256, (
    "Runtime bundle hash mismatch. Use the bundle supplied with this notebook."
)

statuses = {}
with zipfile.ZipFile(RUNTIME_BUNDLE) as archive:
    runtime_manifest = json.loads(
        archive.read("runtime_manifest.json").decode("utf-8")
    )
    assert runtime_manifest["contract_tag"] == CONTRACT_TAG
    SOURCE_HASHES = runtime_manifest["files"]

    for relative_path, expected_hash in sorted(SOURCE_HASHES.items()):
        member = PurePosixPath(relative_path)
        if member.is_absolute() or ".." in member.parts:
            raise ValueError(f"Unsafe bundle member: {relative_path}")
        content = archive.read(relative_path)
        assert hashlib.sha256(content).hexdigest() == expected_hash

        destination = PYTHON_SOURCE_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists():
            if destination.read_bytes() != content:
                raise FileExistsError(
                    f"Immutable artifact differs: {destination}"
                )
            statuses[relative_path] = "verified"
        else:
            destination.write_bytes(content)
            statuses[relative_path] = "created"

print("Runtime bundle:", RUNTIME_BUNDLE)
print("Files created:", sum(value == "created" for value in statuses.values()))
print("Files verified:", sum(value == "verified" for value in statuses.values()))

## 2. Load and inspect the contracts

Validity appears **once**, in `entity_registry.valid_from` / `valid_to`.
There is deliberately no second `entity_service_windows` canonical table.

`gt_condition_states` supports state-labelled datasets such as Petrobras 3W.
It deliberately has no `severity_ordinal`, because 3W does not exercise graded severity.

In [ ]:
if str(PYTHON_SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE_ROOT))

from telemetry_contract import CONTRACT_VERSION
from telemetry_eval_contract import EVAL_CONTRACT_VERSION

schema_root = PYTHON_SOURCE_ROOT / "telemetry_contract" / "schemas" / "spec-core"
eval_schema_root = (
    PYTHON_SOURCE_ROOT / "telemetry_eval_contract" / "schemas" / "spec-eval"
)
SPEC_CORE_TABLES = sorted(path.stem.replace(".schema", "") for path in schema_root.glob("*.schema.json"))
SPEC_EVAL_TABLES = sorted(path.stem.replace(".schema", "") for path in eval_schema_root.glob("*.schema.json"))

assert CONTRACT_VERSION == "0.3.0"
assert EVAL_CONTRACT_VERSION == "0.3.0"
assert "entity_service_windows" not in SPEC_CORE_TABLES

condition_schema = json.loads(
    (eval_schema_root / "gt_condition_states.schema.json").read_text(encoding="utf-8")
)
assert "severity_ordinal" not in condition_schema["properties"]

print("SPEC-CORE:", SPEC_CORE_TABLES)
print("SPEC-EVAL:", SPEC_EVAL_TABLES)

## 3. Inspect the Telecom Pack phrasebook

The catalogue is intentionally rich enough to drive profiling and resampling:
sampling and aggregation semantics, nullability, censoring, bounds, candidate periods,
context keys, and exposure semantics are part of the interface.

In [ ]:
from dataclasses import asdict
import pandas as pd
from telemetry_packs.telecom import load_pack

pack = load_pack()
metric_rows = []
for metric in pack.metric_specs().values():
    row = asdict(metric)
    row["measurement_kind"] = metric.measurement_kind.value
    row["anomaly_direction"] = metric.anomaly_direction.value
    metric_rows.append(row)

relation_rows = [asdict(item) for item in pack.relation_specs().values()]
METRIC_CATALOGUE = pd.DataFrame(metric_rows).sort_values("metric_id")
RELATION_CATALOGUE = pd.DataFrame(relation_rows).sort_values("relation_type")

display(METRIC_CATALOGUE)
display(RELATION_CATALOGUE)

assert set(METRIC_CATALOGUE["anomaly_direction"]) <= {
    "decrease", "increase", "both", "change"
}
assert "groups_ont" in set(RELATION_CATALOGUE["relation_type"])
assert set(RELATION_CATALOGUE["relation_family"]) == {
    "network_topology", "geographic_membership"
}

## 4. Exposure provenance — physical standard versus frozen generator

These facts must not be silently reconciled:

- Physical GPON reference: 2,488,320,000 downstream bits/s and RS(255,239),
  with 2040 transmitted and 1912 payload bits per codeword.
- Frozen generator mechanism: 2,488,000,000 bits/s and an internal divisor of 1904,
  which cancels when its FEC mean is formed.
- Frozen generator CRC mechanism: a fixed 78,000,000 bits/s reference.

The translator reports **generator opportunity counts**, not a claim that
`fec_count` is a standards-defined corrected-codeword counter. Constant exposure is
validated by formula and dimensions, not by variance.

In [ ]:
EXPOSURE_PROVENANCE = {
    "physical_reference": {
        "standard": "ITU-T G.984.3",
        "downstream_line_rate_bps": 2_488_320_000,
        "reed_solomon": "RS(255,239)",
        "transmitted_bits_per_codeword": 2040,
        "payload_bits_per_codeword": 1912,
    },
    "frozen_generator": {
        "release": "telemetry-synth-4.0.1",
        "fec_line_rate_bps": 2_488_000_000,
        "fec_internal_divisor": 1904,
        "crc_reference_rate_bps": 78_000_000,
        "interpretation": (
            "opportunity counts under the generator mechanism; "
            "not a standards-defined corrected-codeword counter"
        ),
    },
    "translation_rule": {
        "fec_exposure": "2_488_000_000 * cadence_seconds",
        "crc_exposure": "78_000_000 * cadence_seconds",
        "mismatch_policy": "report; do not reconcile",
    },
}
print(json.dumps(EXPOSURE_PROVENANCE, indent=2))

## 5. Boundary and dependency checks

The generic contract contains no telecom branch. The pack contains no truth fields.
Only an adapter may import both the observable and evaluation contracts.

In [ ]:
import ast

generic_text = "\n".join(
    path.read_text(encoding="utf-8").lower()
    for path in (PYTHON_SOURCE_ROOT / "telemetry_contract").rglob("*")
    if path.is_file() and path.suffix in {".py", ".json"}
)
forbidden_sector_terms = ("telecom", "telco", "gpon", "ont_id", "rx_power")
assert not [term for term in forbidden_sector_terms if term in generic_text]

pack_text = "\n".join(
    path.read_text(encoding="utf-8").lower()
    for path in (PYTHON_SOURCE_ROOT / "telemetry_packs").rglob("*")
    if path.is_file() and path.suffix in {".py", ".json"}
)
assert "gt_" not in pack_text

for package in ("telemetry_contract", "telemetry_packs", "telemetry_runtime"):
    imports = []
    for path in (PYTHON_SOURCE_ROOT / package).rglob("*.py"):
        tree = ast.parse(path.read_text(encoding="utf-8"))
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                imports.extend(alias.name for alias in node.names)
            elif isinstance(node, ast.ImportFrom) and node.module:
                imports.append(node.module)
    assert not any(name.startswith("telemetry_eval_contract") for name in imports), (
        package, imports
    )

print("Boundary checks passed.")

## 6. Write the immutable contract manifest

In [ ]:
LEGACY_BASELINE = runtime_manifest["legacy_baseline_descriptor"]
LEGACY_ATTACHMENT_MANIFEST = runtime_manifest["legacy_attachment_manifest"]
CONTRACT_MANIFEST = {
    "contract_tag": CONTRACT_TAG,
    "spec_core_version": CONTRACT_VERSION,
    "spec_eval_version": EVAL_CONTRACT_VERSION,
    "telecom_pack_version": pack.pack_version,
    "source_hashes": SOURCE_HASHES,
    "spec_core_tables": SPEC_CORE_TABLES,
    "spec_eval_tables": SPEC_EVAL_TABLES,
    "exposure_provenance": EXPOSURE_PROVENANCE,
    "legacy_baseline_descriptor": LEGACY_BASELINE,
    "legacy_attachment_manifest": LEGACY_ATTACHMENT_MANIFEST,
}
manifest_text = json.dumps(CONTRACT_MANIFEST, indent=2, sort_keys=True) + "\n"
status = write_immutable_text(CONTRACT_ROOT / "contract_manifest.json", manifest_text)
print(status, CONTRACT_ROOT / "contract_manifest.json")
print("Notebook 02 complete.")